# One-dimensional cosine-potential Bloch bands

**Status:** Maintained calculated results for an idealized bare electron in a prescribed cosine potential. This is not a material-specific band structure or a DFT calculation.

**Prerequisite:** [One-dimensional cosine-potential Bloch bands](../../../../docs/lecturenotes/solidstate/bloch1d/cosinepotential/index.md)

This laboratory reuses the generic `pbc1d` boundary representation through package-owned Bloch Hamiltonians.


## Model

$$H=-\frac{\hbar^2}{2m_e}\frac{d^2}{dx^2}+V_0\cos\left(\frac{2\pi x}{a}\right),$$

with Bloch condition

$$\psi(x+a)=e^{ika}\psi(x).$$

At $V_0=0$, the represented operator reduces exactly to the `bloch1d/freeelectron` finite-difference Hamiltonian. A nonzero Fourier component couples free-electron modes separated by one reciprocal vector and opens the first zone-edge gap.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from scipy.constants import Boltzmann, electron_volt

from projectkoios.physkit.dos.thermal import FermiDiracThermalDensityOfStatesEvaluator
from projectkoios.physkit.potentials.periodic1d.visualization import PeriodicPotential1DVisualizer
from projectkoios.physkit.solidstate.bloch1d.cell import BlochPrimitiveCell1D
from projectkoios.physkit.solidstate.bloch1d.cosinepotential.model import CosinePotentialBloch1D
from projectkoios.physkit.solidstate.bloch1d.freeelectron.model import FreeElectronBloch1D
from projectkoios.physkit.dos1d.model import UniformBandDensityOfStates1D
from projectkoios.physkit.solidstate.visualization.bands import BandStructurePlotData, BandStructureVisualizer
from projectkoios.physkit.dos.visualization import DensityOfStatesPlotData, DensityOfStatesVisualizer
from projectkoios.physkit.thermal.statmech.qm import FermiDiracState
from projectkoios.physkit.units import MatrixQuantity, PhysicalUnit, ScalarQuantity, Unitless, UnitSystem, VectorQuantity

cell = BlochPrimitiveCell1D(
    oriented_length=ScalarQuantity(10.0, UnitSystem.METAL.length_unit),
    unit_system=UnitSystem.METAL,
)
model = CosinePotentialBloch1D(
    cell=cell,
    amplitude=ScalarQuantity(0.5, UnitSystem.METAL.energy_unit),
)
reciprocal_unit = PhysicalUnit("1 / angstrom")


## Prescribed periodic potential

The amplitude is an illustrative model parameter, not a fitted material value.


In [ ]:
fractional_coordinates = VectorQuantity(
    np.linspace(0.0, 1.0, 400, endpoint=False),
    Unitless(),
)
fig, ax = plt.subplots(figsize=(7.2, 3.8))
PeriodicPotential1DVisualizer(model=model.periodic_potential).plot(
    fractional_coordinates=fractional_coordinates,
    ax=ax,
)
ax.set_title("Prescribed cosine potential")
fig.tight_layout()


## Bands and the free-electron reference

Solid curves are the represented cosine-potential eigenvalues. Dashed curves are exact folded free-electron branches. Their difference is the prescribed-potential effect plus finite-difference error.


In [ ]:
zone_edge = np.pi / cell.length.magnitude
k_values = np.linspace(-zone_edge, zone_edge, 161)
band_result = model.solve_bands(
    bloch_wave_numbers=VectorQuantity(k_values, reciprocal_unit),
    point_count=64,
    band_count=5,
)
free_model = FreeElectronBloch1D(cell=cell)
free_modes = tuple((n,) for n in range(-4, 5))
free_rows = []
for k in k_values:
    row = free_model.evaluate_modes(
        mode_indices=free_modes,
        bloch_wave_number=ScalarQuantity(float(k), reciprocal_unit),
    ).energies.magnitude
    free_rows.append(np.sort(row)[:5])
free_bands = MatrixQuantity(
    np.asarray(free_rows),
    band_result.energies.unit,
)
band_plot_data = BandStructurePlotData(
    path_coordinates=VectorQuantity(k_values / zone_edge, Unitless()),
    band_energies=band_result.energies,
)
fig, ax = plt.subplots(figsize=(7.4, 4.8))
BandStructureVisualizer(data=band_plot_data).plot(
    ax=ax,
    reference_band_energies=free_bands,
    path_label=r"$k/(\pi/a)$",
    title="Cosine-potential bands and folded free-electron reference",
)
fig.tight_layout()


## One-dimensional density of states

For explicit state degeneracy $g$,

$$g_{1D}(E)=\frac{g}{2\pi}\sum_n\int_{\mathrm{BZ}}\delta(E-E_n(k))\,dk.$$

The regular full-zone mesh excludes the duplicated right endpoint. With spin degeneracy $g=2$, each complete band integrates to $2/a$ states per length. Gaussian broadening is used only to render the sampled spectrum.


In [ ]:
dos_k_values = np.linspace(-zone_edge, zone_edge, 401, endpoint=False)
dos_band_result = model.solve_bands(
    bloch_wave_numbers=VectorQuantity(dos_k_values, reciprocal_unit),
    point_count=64,
    band_count=5,
)
dos_broadening = ScalarQuantity(0.025, band_result.energies.unit)
dos_energy_grid = VectorQuantity(
    np.linspace(
        float(np.min(dos_band_result.energies.magnitude)) - 6.0 * dos_broadening.magnitude,
        float(np.max(dos_band_result.energies.magnitude)) + 6.0 * dos_broadening.magnitude,
        1600,
    ),
    dos_band_result.energies.unit,
)
dos_model = UniformBandDensityOfStates1D(
    brillouin_zone_measure=ScalarQuantity(2.0 * zone_edge, reciprocal_unit),
    state_degeneracy=2.0,
)
dos_result = dos_model.evaluate_gaussian(
    band_energies=dos_band_result.energies,
    energy_grid=dos_energy_grid,
    broadening=dos_broadening,
)
expected_state_density = 10.0 / cell.length.magnitude
assert np.isclose(dos_result.represented_state_density.magnitude, expected_state_density)
assert np.isclose(
    dos_result.integrated_state_density.magnitude,
    dos_result.represented_state_density.magnitude,
    rtol=2e-6,
)
fig, ax = plt.subplots(figsize=(7.1, 4.4))
DensityOfStatesVisualizer(
    data=DensityOfStatesPlotData.from_gaussian(dos_result)
).plot(ax=ax, title="One-dimensional cosine-potential DOS")
fig.tight_layout()


## Fixed-density chemical potential and thermal DOS

The state weights come from the same endpoint-excluded full-zone normalization as the DOS. The solver evaluates occupations on the original sampled eigenvalues rather than histogram bins or the broadened display grid. The target below fills one complete spin-degenerate band, $2/a$ states per length.

The plotted occupied, unoccupied, and thermally active DOS use the broadened DOS only for visualization. Their numerical integrals need not equal the exact sampled-state target at finite grid extent and broadening.


In [ ]:
K_B_EV = Boltzmann / electron_volt
temperature_values = np.array([50.0, 100.0, 200.0, 300.0, 500.0, 800.0, 1000.0])
target_state_density = ScalarQuantity(
    2.0 / cell.length.magnitude,
    reciprocal_unit,
)
chemical_potential_results = []
for temperature in temperature_values:
    chemical_potential_results.append(
        dos_model.solve_chemical_potential(
            band_energies=dos_band_result.energies,
            thermal_energy=ScalarQuantity(
                float(K_B_EV * temperature),
                dos_band_result.energies.unit,
            ),
            target_state_density=target_state_density,
        )
    )
chemical_potentials = np.array(
    [result.chemical_potential.magnitude for result in chemical_potential_results]
)
assert all(
    abs(result.state_weight_residual.magnitude)
    <= 1e-10 * result.request.total_state_weight.magnitude
    for result in chemical_potential_results
)
temperature_index = int(np.flatnonzero(temperature_values == 300.0)[0])
selected_result = chemical_potential_results[temperature_index]
selected_state = FermiDiracState(
    thermal_energy=selected_result.request.thermal_energy,
    chemical_potential=selected_result.chemical_potential,
)
thermal_dos = FermiDiracThermalDensityOfStatesEvaluator().evaluate(
    density_of_states_result=dos_result,
    state=selected_state,
)
fig, axes = plt.subplots(2, 2, figsize=(11.5, 7.5))
axes[0, 0].plot(temperature_values, chemical_potentials, marker="o")
axes[0, 0].set(
    xlabel="Temperature (K)",
    ylabel="Chemical potential (eV)",
    title=r"Fixed-density $\mu(T)$",
)
axes[0, 0].grid(alpha=0.25)
base_plot = DensityOfStatesPlotData.from_gaussian(dos_result)
DensityOfStatesVisualizer(data=base_plot).plot(
    ax=axes[0, 1], label="available", title="DOS at 300 K"
)
for component in ("occupied", "unoccupied"):
    DensityOfStatesVisualizer(
        data=DensityOfStatesPlotData.from_thermal(thermal_dos, component=component)
    ).plot(ax=axes[0, 1], label=component, title="DOS at 300 K")
axes[0, 1].legend()
axes[1, 0].plot(
    thermal_dos.request.energies.magnitude,
    thermal_dos.thermal_window.magnitude,
)
axes[1, 0].set(
    xlabel="Energy (eV)",
    ylabel=r"$-\partial f/\partial E$ (1/eV)",
    title="Fermi--Dirac thermal window at 300 K",
)
axes[1, 0].grid(alpha=0.25)
DensityOfStatesVisualizer(
    data=DensityOfStatesPlotData.from_thermal(
        thermal_dos, component="thermally_active"
    )
).plot(ax=axes[1, 1], title="Thermally active DOS at 300 K")
fig.tight_layout()
print(
    f"mu(300 K) = {selected_result.chemical_potential.magnitude:.6f} eV; "
    f"target = {target_state_density.magnitude:.6f} {target_state_density.unit.expression}"
)


## First zone-edge gap

For a weak $V_0\cos(2\pi x/a)$ perturbation, the first gap approaches $|V_0|$. The calculation below reports represented values; it does not impose an empirical gap threshold.


In [ ]:
amplitudes = np.linspace(0.0, 1.0, 21)
gaps = []
for amplitude in amplitudes:
    candidate = CosinePotentialBloch1D(
        cell=cell,
        amplitude=ScalarQuantity(float(amplitude), UnitSystem.METAL.energy_unit),
    )
    edge = candidate.solve_bands(
        bloch_wave_numbers=VectorQuantity(np.array([zone_edge]), reciprocal_unit),
        point_count=80,
        band_count=2,
    ).energies.magnitude[0]
    gaps.append(edge[1] - edge[0])
fig, ax = plt.subplots(figsize=(6.8, 4.3))
ax.plot(amplitudes, gaps, marker="o", markersize=3, label="represented gap")
ax.plot(amplitudes, amplitudes, linestyle="--", color="black", label=r"weak-potential $|V_0|$")
ax.set(xlabel=r"$V_0$ (eV)", ylabel="First gap (eV)", title="Zone-edge gap versus cosine amplitude")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


## Grid convergence

The zone-edge gap and individual energies have different discretization behavior. This plot checks the represented gap without claiming material convergence.


In [ ]:
point_counts = np.array([12, 16, 24, 32, 48, 64, 96])
represented_gaps = []
for point_count in point_counts:
    edge = model.solve_bands(
        bloch_wave_numbers=VectorQuantity(np.array([zone_edge]), reciprocal_unit),
        point_count=int(point_count),
        band_count=2,
    ).energies.magnitude[0]
    represented_gaps.append(edge[1] - edge[0])
fig, ax = plt.subplots(figsize=(6.8, 4.2))
ax.plot(point_counts, represented_gaps, marker="o")
ax.axhline(model.amplitude.magnitude, linestyle="--", color="black", label=r"weak-potential $|V_0|$")
ax.set(xlabel="Grid points per cell", ylabel="First gap (eV)", title="Represented gap under grid refinement")
ax.legend()
ax.grid(alpha=0.25)
fig.tight_layout()


## Interpretation and retained exploratory work

The package-backed calculation establishes the documented cosine-potential software model and its numerical behavior. It does not establish a semiconductor band gap.

The maintained weighted solver and thermal-DOS result supersede the extended prototype's finite-temperature chemical-potential, occupied/unoccupied DOS, and thermal-window calculations without inheriting its endpoint duplication, histogram-bin inversion, or arbitrary state normalization. The exploratory artifact remains preserved pending its final individual disposition.

## Exercises

1. Verify equality of the spectra at $k=-\pi/a$ and $k=\pi/a$.
2. Explain why changing the sign of $V_0$ leaves the complete spectrum unchanged.
3. Compare individual band-energy convergence with gap convergence.
4. Add a second Fourier harmonic through a new explicit potential contract.
5. Vary the full-zone sample count and Gaussian broadening independently, while checking that the represented state-density integral remains $10/a$.
6. Compare the exact sampled-energy chemical potential with one obtained by integrating a broadened or histogram DOS, and quantify the representation error.
